# Model Development & Hyperparameter Tuning

This notebook implements four classification models:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. XGBoost

Each model will be trained using:
- Proper preprocessing pipelines
- Stratified cross-validation
- Hyperparameter tuning (GridSearchCV)
- Experiment tracking using MLflow

The objective is to identify the most suitable model for credit risk prediction from a business and statistical perspective.

In [ ]:
!pip install mlflow xgboost --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 808.4/808.4 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 14.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Evaluation
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# MLflow
import mlflow
import mlflow.sklearn

import warnings
warnings.filterwarnings("ignore")

SEED = 42

Load Train/Test Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
data_path = "/content/drive/MyDrive/T2_Project_Srinivas Musunuri/Data/"

X_train = pd.read_csv(data_path + "X_train.csv")
X_test = pd.read_csv(data_path + "X_test.csv")
y_train = pd.read_csv(data_path + "y_train.csv").values.ravel()
y_test = pd.read_csv(data_path + "y_test.csv").values.ravel()

print("Data loaded successfully.")

Data loaded successfully.


Recreate Preprocessing Pipeline

In [ ]:
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

numeric_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

Configure MLflow Tracking

In [ ]:
mlflow.set_tracking_uri("file:///content/drive/MyDrive/T2_Project_Srinivas Musunuri/mlruns")
mlflow.set_experiment("Credit_Risk_Model_Comparison")

print("MLflow tracking configured.")

2026/02/21 15:40:31 INFO mlflow.tracking.fluent: Experiment with name 'Credit_Risk_Model_Comparison' does not exist. Creating a new experiment.


MLflow tracking configured.


Define Cross-Validation Strategy

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

Create Model Dictionary

In [ ]:
models = {
    "Logistic_Regression": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED),
    "Decision_Tree": DecisionTreeClassifier(class_weight='balanced', random_state=SEED),
    "Random_Forest": RandomForestClassifier(class_weight='balanced', random_state=SEED),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=SEED)
}

Define Hyperparameter Grids

In [ ]:
param_grids = {
    "Logistic_Regression": {
        "model__C": [0.01, 0.1, 1, 10]
    },
    "Decision_Tree": {
        "model__max_depth": [None, 5, 10, 20],
        "model__min_samples_split": [2, 5, 10]
    },
    "Random_Forest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 10, 20]
    },
    "XGBoost": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.01, 0.1]
    }
}

Training Loop with MLflow Logging

In [ ]:
best_models = {}

for model_name in models:

    with mlflow.start_run(run_name=model_name):

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", models[model_name])
        ])

        grid_search = GridSearchCV(
            pipeline,
            param_grids[model_name],
            cv=cv_strategy,
            scoring="roc_auc",
            n_jobs=-1
        )

        grid_search.fit(X_train, y_train)

        best_model = grid_search.best_estimator_
        best_models[model_name] = best_model

        y_pred = best_model.predict(X_test)
        y_proba = best_model.predict_proba(X_test)[:, 1]

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_proba)

        # Log parameters
        mlflow.log_params(grid_search.best_params_)

        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("roc_auc", roc_auc)

        # Log model
        mlflow.sklearn.log_model(best_model, model_name)

        print(f"{model_name} completed.")

2026/02/21 15:45:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/21 15:45:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logistic_Regression completed.


2026/02/21 15:45:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/21 15:45:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Decision_Tree completed.


2026/02/21 15:45:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/21 15:45:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random_Forest completed.


2026/02/21 15:45:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/21 15:45:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


XGBoost completed.


Save Best Models as .pkl

In [ ]:
import joblib

model_save_path = "/content/drive/MyDrive/T2_Project_Srinivas Musunuri/Models/"

for name, model in best_models.items():
    joblib.dump(model, model_save_path + f"{name}.pkl")

print("All models saved successfully.")

All models saved successfully.
